# Student Health Risk — Playground Series S6E7 · **Notebook finale**

**Task**: classificazione multi-classe (`fit` / `at-risk` / `unhealthy`) · **Metrica**: balanced accuracy
**Dataset**: 690.088 train / ~148k test · 7 numeriche + 6 categoriche · null (~1–12%) su tutte le feature
**Risultato finale**: OOF BA **0.94985** (v3b) · LB v3 **0.94961**
**Ricetta finale**: LightGBM (Optuna) · NaN crudi · prior-correction generalizzata (moltiplicatori per classe)

---

## Post-mortem — le lezioni della campagna

1. **La prior-correction è stata l'intervento con il miglior rapporto resa/costo dell'intera campagna**: da 0.87626 (argmax nudo) a 0.94938, **+7.3 punti** di BA con una divisione e zero retraining. Con metrica balanced accuracy e sbilanciamento 86/8/6, la regola di decisione vale più di qualsiasi tuning: argmax p(c|x)/prior^t, con t*=1.00 (probabilità LightGBM perfettamente calibrate, confermato 3 volte).
2. **L'OOF su 690k righe è l'autorità; il public LB è un sondaggio con margine d'errore ±0.001** (misurato: v3 valeva +0.0004 OOF ma +0.0037 LB; v5 +0.00002 OOF e −0.00001 LB). Ogni decisione presa sull'OOF, mai sul public.
3. **Plateau dimostrato per esaurimento delle tre leve**: *rappresentazione* (alberi LGB/CB + rete MLP: ~40k errori condivisi da tre geometrie diverse), *regola di decisione* (moltiplicatori liberi: superficie piatta attorno alla correzione bayesiana pura), *funzione obiettivo* (pesi bilanciati: −0.0003). Le ~40k righe errate sono rumore del generatore sintetico, non un gap di modeling.
4. **Diversità ≠ complementarità**: l'MLP dissente da LightGBM sull'1.15% delle righe (accordo 0.9885, il piu' "diverso" di tutti)… e dove dissente ha quasi sempre torto. Blend: w*(lgb)=1.0, +0.00000, identico in media aritmetica e geometrica.
5. **Training naturale + correzione ≥ pesi bilanciati + correzione** (domanda aperta da S6E6, chiusa: −0.0003 per i pesi bilanciati). I pesi distorcono la calibrazione durante il training per ottenere ciò che la correzione ottiene dopo, gratis.
6. **Lezione del bordo-griglia**: se l'ottimo di una grid search siede sul confine dello spazio di ricerca, hai trovato il muro, non l'ottimo (v9: m=0.30 su griglia [0.3, 1.8]).
7. **Diagnostica della missingness in tre righe prima di qualsiasi imputation**: qui ≈ MCAR ovunque → zero imputation (gli split direzionali di LightGBM sono un'imputation appresa). E conteggi marginali identici ≠ stesse righe (overlap 72 ≈ 69 attesi sotto indipendenza).
8. **Strade rifiutate consapevolmente**: arbitraggio dei file pubblici (ensemble del lavoro altrui — legale, non formativo) e tuning LB-guided (delta di +0.0001 "validati" su uno strumento con risoluzione ±0.001).
9. **Resa di Optuna su una baseline sana: +0.0004.** Sapere quando il tuning è finito vale quanto saperlo fare.

---

## Log delle versioni — completo

| ver | descrizione | OOF BA (post-correzione) | LB | verdetto |
|-----|-------------|--------------------------|-----|----------|
| v1  | LightGBM baseline, NaN crudi, prior-correction t=1.00 | 0.94938 | 0.94591 ✓ | ancoraggio OOF↔LB verificato |
| v2  | + `bmi_is_missing` | = v1 (delta 0.00000) | — | scartata |
| v3  | Optuna LGB (obiettivo BA post-correzione), t*=1.00 | 0.94979 | 0.94961 ✓ | **modello finale** |
| v4  | CatBoost GPU, parametri sobri | 0.94888 | — | inferiore, accordo 0.9929 |
| v5  | Blend LGB+CB | 0.94981, w*(lgb)=0.95 | 0.94960 | delta nel rumore |
| v6  | Optuna CatBoost GPU | 0.94923 | — | accordo *salito* a 0.9942; blend w*=1.0 |
| v7  | 6 interazioni numeriche (ablation) | = v3 | — | 0 sopravvissute allo screening |
| v8  | MLP + embedding (GPU) | 0.94762, t*=0.90 | — | accordo 0.9885 ma blend +0.00000: diversità senza complementarità |
| v3b | Moltiplicatori liberi (m_fit=1.33, m_unh=1.02) | **0.94985** | — | **submission finale** (+0.00007, entro il rumore, costo zero) |
| v9  | `class_weight='balanced'` + correzione | 0.94948 | — | −0.0003: la loss naturale vince |

*Errori condivisi sul nocciolo duro: lgb∩cb = 40.249 · lgb∩mlp = 39.632 (su ~42–45k totali per modello).*

## 1. Setup e dati

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from itertools import product
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
import warnings
warnings.filterwarnings('ignore')

import kagglehub
kagglehub.competition_download('playground-series-s6e7')
BASE = '/kaggle/input/competitions/playground-series-s6e7'

SEED, N_FOLDS = 42, 5
train = pd.read_csv(f'{BASE}/train.csv')
test  = pd.read_csv(f'{BASE}/test.csv')
print(train.shape, test.shape)

## 2. EDA essenziale e diagnostica della missingness

Target 86/8/6 (`at-risk`/`unhealthy`/`fit`): con la balanced accuracy, il campo di battaglia sono le
minoritarie. Missingness ≈ MCAR ovunque (unica anomalia: `bmi`, 0.71% null in `unhealthy` vs 2.86%
in `fit` — testata come feature in v2, delta zero: LightGBM vede già il NaN negli split).

In [ ]:
display(train['health_condition'].value_counts(normalize=True).round(4))

msno = train.drop(columns=['id', 'health_condition']).isnull()
display(train.groupby('health_condition')[msno.columns]
             .apply(lambda g: g.isnull().mean()).T.round(4))

## 3. Preprocessing minimale e pipeline

- NaN numerici crudi (split direzionali = imputation appresa) · categoriche a `category` (NaN = categoria)
- StratifiedKFold 5, seed 42 — gli stessi fold per ogni esperimento della campagna
- `run_cv` + `prior_sweep`: la coppia riusabile che ha garantito confronti puliti per 10 versioni

In [ ]:
TARGET = 'health_condition'
cat_cols = train.select_dtypes('object').columns.drop(TARGET).tolist()

le = LabelEncoder()
y = le.fit_transform(train[TARGET])
IDX_FIT = int(le.transform(['fit'])[0])
IDX_UNH = int(le.transform(['unhealthy'])[0])

def make_X(df):
    X_ = df.drop(columns=['id', TARGET], errors='ignore').copy()
    for c in cat_cols:
        X_[c] = X_[c].astype('category')
    return X_

X, X_test = make_X(train), make_X(test)
priors = np.bincount(y) / len(y)

def run_cv(X_, y, params, n_folds=N_FOLDS, seed=SEED, label=''):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    oof, models = np.zeros((len(X_), 3)), []
    for fold, (tr, va) in enumerate(skf.split(X_, y)):
        m = lgb.LGBMClassifier(**params)
        m.fit(X_.iloc[tr], y[tr], eval_set=[(X_.iloc[va], y[va])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[va] = m.predict_proba(X_.iloc[va])
        models.append(m)
        print(f'[{label}] fold {fold}: BA = {balanced_accuracy_score(y[va], oof[va].argmax(1)):.5f}')
    print(f'[{label}] OOF BA (argmax): {balanced_accuracy_score(y, oof.argmax(1)):.5f}')
    return oof, models

def prior_sweep(oof, y, priors, ts=np.arange(0.85, 1.16, 0.05)):
    results = {round(float(t), 2): balanced_accuracy_score(y, (oof / priors**t).argmax(1)) for t in ts}
    t_star = max(results, key=results.get)
    print(f't* = {t_star:.2f}, BA = {results[t_star]:.5f}')
    return t_star, results

## 4. Modello finale — LightGBM v3 (Optuna)

⚠️ **Riproducibilità**: incollare qui sotto i parametri esatti da `study.best_params` della run
Optuna originale (sezione 8 del notebook di sviluppo). I valori sotto sono i base params;
senza quelli del tuning si riproduce la v1 (0.94938), non la v3 (0.94979).

In [ ]:
BEST_PARAMS = dict(
    objective='multiclass', num_class=3,
    n_estimators=3000, random_state=SEED, verbosity=-1, bagging_freq=1,
    # --- TODO: incollare study.best_params della run Optuna v3 ---
    learning_rate=0.05, num_leaves=63,
)

oof, models = run_cv(X, y, BEST_PARAMS, label='final')
t_star, sweep = prior_sweep(oof, y, priors)

## 5. Regola di decisione finale — moltiplicatori liberi (v3b)

Generalizzazione della prior-correction: argmax di (p/prior)·m con m = (m_fit, 1, m_unhealthy)
tunati coarse→fine sull'OOF. Risultato storico: m_fit=1.330, m_unhealthy=1.020, BA 0.94985
(+0.00007 sul t-sweep: entro il rumore, adottato perché a costo zero). La superficie attorno
alla correzione bayesiana pura è piatta — la firma di probabilità ben calibrate.

In [ ]:
base = oof / priors

def mult_search(grid_f, grid_u):
    best = (-1, 1.0, 1.0)
    for mf, mu in product(grid_f, grid_u):
        m = np.ones(3); m[IDX_FIT] = mf; m[IDX_UNH] = mu
        ba = balanced_accuracy_score(y, (base * m).argmax(1))
        if ba > best[0]:
            best = (ba, mf, mu)
    return best

ba_c, mf_c, mu_c = mult_search(np.linspace(0.7, 1.6, 19), np.linspace(0.7, 1.6, 19))
ba_f, mf_star, mu_star = mult_search(np.linspace(mf_c-0.05, mf_c+0.05, 11),
                                     np.linspace(mu_c-0.05, mu_c+0.05, 11))
assert not np.isclose(mf_star, [0.7, 1.6]).any(), 'ottimo sul bordo griglia: allargare!'
print(f'v3b: BA = {ba_f:.5f} con m_fit={mf_star:.3f}, m_unhealthy={mu_star:.3f}')

## 6. Submission finale

In [ ]:
m_star = np.ones(3); m_star[IDX_FIT] = mf_star; m_star[IDX_UNH] = mu_star

test_proba = np.mean([m.predict_proba(X_test) for m in models], axis=0)
test_pred  = le.classes_[((test_proba / priors) * m_star).argmax(1)]

sub = pd.DataFrame({'id': test['id'], TARGET: test_pred})
sub.to_csv('submission.csv', index=False)
print(sub[TARGET].value_counts(normalize=True).round(4))
sub.head()

## Appendice — Il cimitero degli esperimenti (con onore)

Ogni lapide qui sotto è un'ipotesi uccisa da un'ablation pre-dichiarata — il contrario del tempo perso.

- **v2 · `bmi_is_missing`** — unica anomalia di missingness del dataset; delta 0.00000: gli split
  direzionali la catturavano già. *Lezione: l'indicatore esplicito paga solo se il modello non vede il NaN.*
- **v4/v6 · CatBoost (sobrio e tunato)** — 0.94888 → 0.94923, ma accordo con lgb salito da 0.9929
  a 0.9942 e blend con w*(lgb)=1.0. *Lezione: il tuning indipendente fa convergere modelli diversi
  verso lo stesso confine bayesiano: oltre c'è solo rumore.*
- **v5 · blend LGB+CB** — +0.00002 con w*=0.95. Tetto teorico calcolabile a priori dagli errori
  esclusivi (1.825 vs 2.856 su 40.249 condivisi). *Lezione: la diagnostica di diversità PRIMA del blend.*
- **v7 · 6 interazioni numeriche** — delta da −0.00012 a +0.00002, zero sopravvissute. *Lezione: con
  690k righe gli alberi si costruiscono da soli le interazioni che contano.*
- **v8 · MLP + embedding** — 0.94762, accordo 0.9885 (il più diverso di tutti), blend +0.00000 in
  aritmetica E geometrica. 39.632 errori condivisi con lgb. *Lezione: diversità senza complementarità;
  e le reti richiedono l'imputation (mediana + flag, statistiche per fold) che agli alberi non serve.*
- **v9 · pesi bilanciati** — 0.94948, −0.0003 vs naturale+correzione. Chiusa la domanda aperta da
  S6E6. *Lezione bonus: ottimo di grid search sul bordo della griglia = muro, non ottimo.*
- **Non tentati per scelta**: arbitraggio dei file pubblici e tuning LB-guided — vedi post-mortem, punto 8.